In [0]:
%sql

-- Create customer segments based on country (as a proxy for Enterprise/SMB/Consumer)
-- Using country as the primary segmentation dimension
SELECT 
  CASE 
    WHEN c.`country` = 'United States' THEN 'North America'
    WHEN c.`country` IN ('United Kingdom', 'France', 'Germany') THEN 'Europe'
    WHEN c.`country` IN ('Australia', 'Canada') THEN 'Pacific & Canada'
    WHEN c.`country` = 'n/a' THEN 'Unknown/Online'
    ELSE 'Other Markets'
  END AS customer_segment,
  SUM(f.`sales_amount`) AS total_sales_amount,
  COUNT(DISTINCT f.`order_number`) AS order_count,
  COUNT(DISTINCT f.`customer_key`) AS customer_count,
  ROUND(100.0 * SUM(f.`sales_amount`) / SUM(SUM(f.`sales_amount`)) OVER (), 2) AS sales_percentage
FROM `workspace`.`gold`.`fact_sales` f
JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`
WHERE f.`order_date` IS NOT NULL
  AND c.`country` IS NOT NULL
GROUP BY 
  CASE 
    WHEN c.`country` = 'United States' THEN 'North America'
    WHEN c.`country` IN ('United Kingdom', 'France', 'Germany') THEN 'Europe'
    WHEN c.`country` IN ('Australia', 'Canada') THEN 'Pacific & Canada'
    WHEN c.`country` = 'n/a' THEN 'Unknown/Online'
    ELSE 'Other Markets'
  END
ORDER BY total_sales_amount DESC
